<a href="https://colab.research.google.com/github/agil-p-varghese/Fine-Tunning-Gemma-By-LORA-Technique/blob/main/Fine_Tuning_Gemma_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip3 install -q -U accelerate bitsandbytes datasets peft transformers trl huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 57.2 MB/s eta 0:00:00


In [ ]:
import os
import torch
import transformers
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Load Hugging Face token from Colab secrets
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

model_id = "google/gemma-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ['HF_TOKEN'])
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ['HF_TOKEN']
)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [ ]:
#Before Training
text = "Quote: Imagination is more,"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Quote: Imagination is more, than knowledge.

I am a self-taught artist, born in 1985 in the beautiful city of Porto Alegre, Brazil.

I have a degree in Fine Arts from the University of Passo Fundo, in the state of Rio Grande do Sul.

I have been painting since I was a child, but I only started to take it seriously in 2015.

I am a self-taught artist, but I have a degree in Fine Arts from the University


In [ ]:
os.environ["WANDB_DISABLED"] = "true"

lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

data = load_dataset("Abirate/english_quotes")

def format_dataset(example):
    return {"text": f"Quote: {example['quote']}\nAuthor: {example['author']}"}

data['train'] = data['train'].map(format_dataset)

In [ ]:
# 2. Initialize Trainer with optimized training parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=data['train'],
    peft_config=lora_config,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=300,            # <--- Increased to 300 for better learning
        learning_rate=1e-4,       # <--- Lowered from 2e-4 to smooth out loss spikes
        bf16=True,                # <--- Matches bfloat16 quantization setup
        logging_steps=5,          # <--- Logs every 5 steps to keep output clean
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
)

# 3. Run the updated training loop
trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
5,2.635045
10,2.776360
15,2.116799
20,2.042637
25,2.162095
30,1.881250
35,2.119006
40,2.242945
45,1.788922
50,1.762779


TrainOutput(global_step=300, training_loss=1.7631580495834351, metrics={'train_runtime': 1120.3893, 'train_samples_per_second': 1.071, 'train_steps_per_second': 0.268, 'total_flos': 720784731095040.0, 'train_loss': 1.7631580495834351, 'epoch': 0.4784688995215311})

In [ ]:
# 1. Save the model adapters and tokenizer to a folder
output_dir = "gemma-quotes-lora"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved successfully to {output_dir}!")

# 2. Zip the folder so it's easy to download
!zip -r gemma-quotes-lora.zip gemma-quotes-lora

# 3. Trigger automatic download in your browser
from google.colab import files
files.download('gemma-quotes-lora.zip')

Model saved successfully to gemma-quotes-lora!
  adding: gemma-quotes-lora/ (stored 0%)
  adding: gemma-quotes-lora/adapter_config.json (deflated 61%)
  adding: gemma-quotes-lora/README.md (deflated 65%)
  adding: gemma-quotes-lora/adapter_model.safetensors (deflated 21%)
  adding: gemma-quotes-lora/tokenizer_config.json (deflated 50%)
  adding: gemma-quotes-lora/tokenizer.json (deflated 84%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#After Training
text = "Quote: A woman is like a tea bag;"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Quote: A woman is like a tea bag; you                                                                                                  


In [ ]:
text = "Quote: A woman is like a tea bag; you"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=60,       # Keep it shorter so it doesn't have room to loop
    do_sample=True,
    temperature=0.3,         # <--- Lower temperature for more focused text
    top_p=0.85,
    repetition_penalty=1.3,  # <--- Higher penalty to strictly block word repeats
    eos_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Quote: A woman is like a tea bag; you can are  P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P


In [6]:
from google.colab import files

uploaded = files.upload()

Saving gemma-quotes-lora.zip to gemma-quotes-lora.zip


In [7]:
import os

print(os.listdir("/content"))

['.config', 'gemma-quotes-lora.zip', 'sample_data']


In [1]:
!pip install -q -U bitsandbytes accelerate peft transformers

In [9]:
import os
import zipfile
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel


# ============================================================
# 1. ZIP FILE
# ============================================================

zip_filename = "/content/gemma-quotes-lora.zip"
extract_path = "/content/gemma-quotes-lora"

if not os.path.exists(zip_filename):
    raise FileNotFoundError(
        f"Could not find {zip_filename}. "
        "Please upload gemma-quotes-lora.zip to Colab first."
    )

print("Extracting LoRA adapter...")

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")


# ============================================================
# 2. FIND adapter_config.json
# ============================================================

adapter_config_path = None

for root, dirs, files in os.walk(extract_path):
    if "adapter_config.json" in files:
        adapter_config_path = os.path.join(root, "adapter_config.json")
        break

if adapter_config_path is None:
    raise FileNotFoundError(
        "adapter_config.json was not found inside the ZIP file."
    )

adapter_path = os.path.dirname(adapter_config_path)

print("LoRA adapter found at:")
print(adapter_path)


# ============================================================
# 3. BASE MODEL
# ============================================================

model_id = "google/gemma-2b"

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading base Gemma model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


# ============================================================
# 4. ATTACH YOUR LoRA ADAPTER
# ============================================================

print("\nLoading fine-tuned LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("LoRA adapter successfully loaded!")


# ============================================================
# 5. TEST GENERATION
# ============================================================

prompt = "Rowling"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.3,
        repetition_penalty=1.4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

print("\n--- Output ---")

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(generated_text)

Extracting LoRA adapter...
Extraction complete!
LoRA adapter found at:
/content/gemma-quotes-lora/gemma-quotes-lora

Loading tokenizer...
Loading base Gemma model...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]


Loading fine-tuned LoRA adapter...
LoRA adapter successfully loaded!

--- Output ---
Rowling’s first book, <em>Harry Potter and the Philosopher’s Stone</em> (1997), was published in Britain under the title <em>Philosopher's Stone</em>. The name of the novel is a reference to Harry being raised by his aunt and uncle after his parents died when


In [17]:
# ============================================================
# TEST MULTIPLE PROMPTS
# ============================================================

prompts = [
    # --- Part 1: From / Similar to the Training Dataset ---
    "Quote: Be the change that you wish to see",
    "Quote: Two things are infinite: the universe and human stupidity;",
    "Quote: In three words I can sum up everything I've learned about life:",

    # --- Part 2: Outside the Dataset (Novel / Generalization Test) ---
    "Quote: Artificial intelligence is like a tool;",
    "Quote: The best way to predict the future is to",
    "Quote: Coffee is the secret fuel that"
]

for i, prompt in enumerate(prompts, 1):

    print(f"\n{'='*60}")
    print(f"PROMPT {i}")
    print(f"{'='*60}")
    print(prompt)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.3,
            repetition_penalty=1.4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\nOUTPUT:")
    print(generated_text)


PROMPT 1
Quote: Be the change that you wish to see

OUTPUT:
Quote: Be the change that you wish to see in the world.
Author: Mahatma Gandhi

PROMPT 2
Quote: Two things are infinite: the universe and human stupidity;

OUTPUT:
Quote: Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.
Author: Albert Einstein

PROMPT 3
Quote: In three words I can sum up everything I've learned about life:

OUTPUT:
Quote: In three words I can sum up everything I've learned about life: it goes on.
Author: Jack Kerouac

PROMPT 4
Quote: Artificial intelligence is like a tool;

OUTPUT:
Quote: Artificial intelligence is like a tool; it’s not the end goal. The real value of AI comes from how you use that technology to make better decisions, build more robust systems and improve people's lives.
Author: Jeff Bezos

PROMPT 5
Quote: The best way to predict the future is to

OUTPUT:
Quote: The best way to predict the future is to create it.
Author: Abraham Lincoln

PROMPT 6